# 02 — Preprocessing e Feature Engineering

## Obiettivi didattici

1. Costruire una pipeline `sklearn` **resistente al data leakage**.
2. Distinguere encoding **ordinale** vs **one-hot** e applicare ognuno alle colonne corrette.
3. Aggiungere feature derivate domain-specific (TotalSF, HouseAge, QualityScore, ...).
4. Verificare la composizione del preprocessor (numero di feature in uscita, sparsità, ...).


In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from ames_pipeline.data import load_raw
from ames_pipeline.wrangling import fill_structural_missing, remove_grliv_area_outliers
from ames_pipeline.features import AmesFeatureEngineer
from ames_pipeline.preprocessing import build_preprocessor, infer_column_groups


## Step 1 — Wrangling (no leakage)

Le operazioni che possono essere fatte fuori dalla pipeline sklearn sono SOLO quelle che non dipendono da statistiche del training set. In Ames sono due:

- `fill_structural_missing`: mappatura semantica di NaN (no statistiche).
- `remove_grliv_area_outliers`: rimozione di record fissi (no statistiche).

Tutto il resto (imputazione mediana, scaling, encoding) **deve** stare dentro la pipeline.

In [ ]:
df = load_raw()
df = fill_structural_missing(df)
df = remove_grliv_area_outliers(df)
X = df.drop(columns=['Order','PID','SalePrice'])
y = df['SalePrice']
print(f'X.shape={X.shape}, y.shape={y.shape}')


## Step 2 — Feature engineering custom

Creiamo feature derivate **interpretabili**:

| Nuova feature        | Definizione                                  | Razionale |
|----------------------|----------------------------------------------|-----------|
| `HouseAge`           | YrSold - YearBuilt                           | Età al momento della vendita. |
| `YearsSinceRemodel`  | YrSold - YearRemodAdd                        | Modernità della ristrutturazione. |
| `TotalSF`            | 1stFlrSF + 2ndFlrSF + TotalBsmtSF            | Area totale abitabile. |
| `TotalBath`          | full + 0.5·half (sopra + sotto)              | Conteggio bagni pesato. |
| `QualityScore`       | OverallQual × OverallCond                    | Interazione qualità×condizione. |
| `HasGarage/Pool/...` | binarie da area > 0                           | Catturano presenza/assenza. |
| `AreaPerRoom`        | GrLivArea / (TotRmsAbvGrd + 1)               | Densità abitativa. |

Le feature derivate aiutano soprattutto i **modelli lineari**, che faticano a modellare interazioni e rapporti non lineari. RF/XGB le imparerebbero comunque, ma renderle esplicite migliora interpretabilità e velocità.

In [ ]:
fe = AmesFeatureEngineer()
X_fe = fe.fit_transform(X)
new_cols = sorted(set(X_fe.columns) - set(X.columns))
print(f'Feature aggiunte ({len(new_cols)}):')
for c in new_cols:
    print(f'  - {c}')


## Step 3 — ColumnTransformer

Il preprocessor è un `ColumnTransformer` con tre branch parallele:

```
   ┌── numeriche  → SimpleImputer(median)
   ├── ordinali   → SimpleImputer('None') + OrdinalEncoder(categories=...)
   └── nominali   → SimpleImputer('None') + OneHotEncoder(handle_unknown='ignore')
```

**Perché branch separate?** Ognuna richiede una strategia diversa di imputazione/encoding. Il `ColumnTransformer` le combina mantenendo la concatenazione corretta delle feature in uscita.

In [ ]:
groups = infer_column_groups(X_fe)
preprocessor = build_preprocessor(
    numeric_cols=groups['numeric'],
    ordinal_cols=groups['ordinal'],
    nominal_cols=groups['nominal'],
)
preprocessor

## Step 4 — Verifica forma di output

Il `OneHotEncoder` espande le categoriche nominali in dummy. Vediamo il numero finale di feature e quante sono dummy.

In [ ]:
X_transformed = preprocessor.fit_transform(X_fe)
n_in = X_fe.shape[1]
n_out = X_transformed.shape[1]
print(f'Feature in:  {n_in}')
print(f'Feature out: {n_out}  (espansione {(n_out / n_in - 1) * 100:.0f}% dovuta al OneHotEncoder)')
print(f'Densità: {(X_transformed != 0).mean():.2%}')


## Conclusione

Il preprocessor è una funzione pura — `fit` su training, `transform` su qualsiasi nuovo dato. Lo passiamo al notebook successivo come step iniziale di tre pipeline candidate.